# Stage 2: Facial Image Preprocessing

This notebook prepares the group's real participant images for facial-recognition experiments.

## Participant labels
- Ella → participant_01
- King → participant_02
- Nelly → participant_03
- Sarah → participant_04

**Data requirement:** Download the shared Google Drive images into `data/images/raw/` before running this notebook. Preserve participant/expression subfolders where possible. The notebook does not invent images or identities.

Supported folder examples:
```text
data/images/raw/
  Ella/neutral/*.jpg
  Ella/smiling/*.jpg
  Ella/surprised/*.jpg
  King/neutral/*.jpg
  ...
```
It also recognizes `participant_01`, `participant_02`, etc. Expression names can be in folder names or filenames.

## 1. Python 3.14.6 setup and imports

This notebook is intended to run with **Python 3.14.6** and OpenCV 5. OpenCV 5 moved the Haar cascade API out of the old top-level `cv2` namespace, so the face-detection cell below checks the OpenCV 5 `xobjdetect` namespace and remains compatible with older OpenCV releases.

If the required packages are not installed, run these commands in the VS Code terminal using the same Python 3.14.6 interpreter:

```powershell
python -m pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
python -m pip install --only-binary=:all: numpy pandas matplotlib scikit-image tqdm opencv-contrib-python==5.0.0.93
```

The first command removes overlapping OpenCV wheel packages. The second installs only the contrib build, which contains the OpenCV 5 `xobjdetect` module. `--only-binary=:all:` prevents pip from attempting a local NumPy source build. Restart the notebook kernel after installation.

In [ ]:
import sys
from pathlib import Path
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

if sys.version_info[:2] != (3, 14):
    print("WARNING: This notebook is configured for Python 3.14.x. Select your Python 3.14.6 interpreter in VS Code.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "images" / "raw"
PROCESSED_DIR = ROOT / "data" / "images" / "processed"
AUGMENTED_DIR = PROCESSED_DIR / "augmented"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

PARTICIPANTS = {
    "ella": "participant_01", "participant_01": "participant_01",
    "king": "participant_02", "participant_02": "participant_02",
    "nelly": "participant_03", "participant_03": "participant_03",
    "sarah": "participant_04", "participant_04": "participant_04",
}
EXPRESSIONS = {"neutral", "smiling", "surprised"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Project root:", ROOT)
print("Raw image folder:", RAW_DIR)
print("Raw folder exists:", RAW_DIR.exists())

## 2. Find images and identify participant/expression labels

The script searches folder names and filenames. Review the resulting table carefully. Any file whose participant or expression cannot be identified is reported and excluded rather than assigned a guessed label.

In [ ]:
def tokens_from_path(path):
    # Include all directory names relative to RAW_DIR and the filename stem.
    rel = path.relative_to(RAW_DIR)
    return [part.lower() for part in rel.parts[:-1]] + [path.stem.lower()]

def identify_participant(path):
    tokens = tokens_from_path(path)
    joined = "_".join(tokens)
    for alias, label in PARTICIPANTS.items():
        if re.search(rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])", joined):
            return label
    return None

def identify_expression(path):
    joined = "_".join(tokens_from_path(path))
    matches = [expr for expr in EXPRESSIONS
               if re.search(rf"(?<![a-z0-9]){expr}(?![a-z0-9])", joined)]
    return matches[0] if len(matches) == 1 else None

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Image folder not found: {RAW_DIR}\n"
        "Download the shared Drive images and place them in data/images/raw/."
    )

image_paths = sorted(p for p in RAW_DIR.rglob("*")
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
records, unresolved = [], []
for path in image_paths:
    participant = identify_participant(path)
    expression = identify_expression(path)
    if participant is None or expression is None:
        unresolved.append({"file": str(path.relative_to(RAW_DIR)),
                           "participant_detected": participant,
                           "expression_detected": expression})
    else:
        records.append({"path": path, "participant_id": participant,
                        "expression": expression})

inventory = pd.DataFrame(records)
print("Image files found:", len(image_paths))
print("Images with recognized labels:", len(inventory))
print("Unresolved files:", len(unresolved))
display(inventory.head(20))
if unresolved:
    display(pd.DataFrame(unresolved).head(30))

## 3. Check sample counts and image quality

The assignment asks for neutral, smiling, and surprised samples. Check counts per participant and expression. Images that cannot be decoded or have very small dimensions should be investigated.

In [ ]:
if inventory.empty:
    raise ValueError("No labeled images found. Check folder names and filenames.")

count_table = (inventory.groupby(["participant_id", "expression"])
               .size().unstack(fill_value=0)
               .reindex(columns=sorted(EXPRESSIONS), fill_value=0))
display(count_table)

quality = []
for row in inventory.itertuples():
    img = cv2.imread(str(row.path))
    if img is None:
        quality.append({"file": str(row.path), "readable": False,
                        "width": None, "height": None})
    else:
        h, w = img.shape[:2]
        quality.append({"file": str(row.path), "readable": True,
                        "width": w, "height": h})
quality_df = pd.DataFrame(quality)
display(quality_df.head(20))
print("Unreadable images:", int((~quality_df["readable"]).sum()))

In [ ]:
%pip install --no-cache-dir opencv-contrib-python==4.12.0.88

## 4. Detect and standardize face crops

A Haar cascade is used as a lightweight face detector. The largest detected face is cropped, converted to grayscale, and resized to 128 × 128 pixels. Images where no face is detected are recorded as skipped; do not silently treat a full image as a face.

In [ ]:
# OpenCV 4 exposes CascadeClassifier at cv2.CascadeClassifier.
# OpenCV 5 moves it to the contrib xobjdetect module.
if hasattr(cv2, "CascadeClassifier"):
    CascadeClassifier = cv2.CascadeClassifier
elif hasattr(cv2, "xobjdetect") and hasattr(cv2.xobjdetect, "CascadeClassifier"):
    CascadeClassifier = cv2.xobjdetect.CascadeClassifier
else:
    raise ImportError(
        "The Haar cascade API is unavailable. Install only opencv-contrib-python==5.0.0.93 "
        "in this Python 3.14 environment, restart the kernel, and rerun the notebook. "
        f"Detected OpenCV version: {cv2.__version__}"
    )

cascade_dir = Path(cv2.data.haarcascades)
cascade_path = cascade_dir / "haarcascade_frontalface_default.xml"
if not cascade_path.exists():
    raise FileNotFoundError(
        f"The Haar cascade XML file was not found at {cascade_path}. "
        "Check that the opencv-contrib-python wheel is installed correctly."
    )

face_detector = CascadeClassifier(str(cascade_path))
assert not face_detector.empty(), f"Could not load OpenCV Haar face detector: {cascade_path}"
print("Face detector loaded successfully:", cascade_path)

def extract_face(image_path, size=(128, 128)):
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40)
    )
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    # Add a small margin while keeping crop within image boundaries.
    margin = int(0.15 * max(w, h))
    x0, y0 = max(0, x-margin), max(0, y-margin)
    x1, y1 = min(gray.shape[1], x+w+margin), min(gray.shape[0], y+h+margin)
    crop = gray[y0:y1, x0:x1]
    return cv2.resize(crop, size, interpolation=cv2.INTER_AREA)

face_records, skipped = [], []
for row in tqdm(inventory.itertuples(), total=len(inventory)):
    face = extract_face(row.path)
    if face is None:
        skipped.append({"file": str(row.path), "reason": "Unreadable or no face detected"})
        continue
    face_records.append({
        "source_path": str(row.path),
        "participant_id": row.participant_id,
        "expression": row.expression,
        "face": face
    })

print("Successfully processed faces:", len(face_records))
print("Skipped images:", len(skipped))
if skipped:
    display(pd.DataFrame(skipped).head(30))

## 5. Visualize standardized face crops

Inspect examples from each expression before augmentation. Face detection is not perfect, so manually review the crops.

In [ ]:
if face_records:
    fig, axes = plt.subplots(1, min(6, len(face_records)), figsize=(12, 3))
    axes = np.atleast_1d(axes)
    for ax, item in zip(axes, face_records[:6]):
        ax.imshow(item["face"], cmap="gray")
        ax.set_title(f'{item["participant_id"]}\n{item["expression"]}')
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    raise ValueError("No faces were detected. Review image quality and face orientation.")

## 6. Augment training images

For each detected face, create controlled variants: a small rotation, horizontal flip, and grayscale normalization. Since face crops are already grayscale, the grayscale operation is represented by contrast normalization. These variants increase training variety but **must not be counted as new independent participants or independent original captures**. Keep all variants from one original image in the same train/test split to prevent leakage.

In [ ]:
def rotate_image(img, angle):
    h, w = img.shape
    matrix = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    return cv2.warpAffine(img, matrix, (w, h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT)

augmented_records = []
for idx, item in enumerate(tqdm(face_records)):
    base = item["face"]
    variants = [
        ("original", base),
        ("rotated_plus_10", rotate_image(base, 10)),
        ("rotated_minus_10", rotate_image(base, -10)),
        ("horizontal_flip", cv2.flip(base, 1)),
        ("contrast_normalized", cv2.equalizeHist(base)),
    ]
    for aug_name, image in variants:
        out_name = f'{item["participant_id"]}_{idx:04d}_{item["expression"]}_{aug_name}.png'
        out_path = AUGMENTED_DIR / out_name
        cv2.imwrite(str(out_path), image)
        augmented_records.append({
            "image_path": str(out_path),
            "source_path": item["source_path"],
            "participant_id": item["participant_id"],
            "expression": item["expression"],
            "augmentation": aug_name,
            "image": image
        })

augmented_df = pd.DataFrame([{k:v for k,v in x.items() if k != "image"}
                             for x in augmented_records])
print("Original detected faces:", len(face_records))
print("Augmented images including originals:", len(augmented_records))
display(augmented_df.head())

## 7. Extract HOG image features

Histogram of Oriented Gradients (HOG) summarizes local edge and shape patterns. It provides a reproducible baseline feature representation for the image dataset. It is not a state-of-the-art face embedding; compare it with a pretrained face-embedding approach if time and compute permit.

In [ ]:
from skimage.feature import hog

feature_rows = []
for item in tqdm(augmented_records):
    image = item["image"]
    vector = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    row = {
        "image_path": item["image_path"],
        "source_path": item["source_path"],
        "participant_id": item["participant_id"],
        "expression": item["expression"],
        "augmentation": item["augmentation"],
    }
    row.update({f"hog_{i:04d}": float(value)
                for i, value in enumerate(vector)})
    feature_rows.append(row)

image_features = pd.DataFrame(feature_rows)
print("Feature table shape:", image_features.shape)
display(image_features.iloc[:5, :12])

## 8. Save features and processing metadata

The main output is `image_features.csv`. A separate inventory and skipped-image report make the preprocessing traceable.

In [ ]:
image_features_path = PROCESSED_DIR / "image_features.csv"
inventory_path = PROCESSED_DIR / "image_inventory.csv"
skipped_path = PROCESSED_DIR / "skipped_images.csv"

image_features.to_csv(image_features_path, index=False)
inventory.drop(columns=["path"]).assign(
    path=inventory["path"].astype(str)
).to_csv(inventory_path, index=False)
pd.DataFrame(skipped).to_csv(skipped_path, index=False)

print("Saved:")
print(image_features_path)
print(inventory_path)
print(skipped_path)

## 9. Final checks and limitations

- Confirm all four participant IDs appear in the inventory and feature CSV.
- Confirm all three expressions have genuine source images for each participant where available.
- Inspect skipped images and fix filenames/folders or image quality where necessary.
- Split train/test data by original `source_path` or participant, never randomly by augmented image row. Randomly splitting augmented variants leaks near-identical images into both sets and inflates performance.
- HOG is a baseline feature extractor. A later model notebook should compare suitable classifiers and report accuracy, precision, recall, F1, and a confusion matrix. Face-recognition evaluation should use participant-independent or carefully designed verification splits.

Audio processing is a separate next stage and should not be mixed into this notebook.